In [ ]:
# This notebook writes a damage-marked output video.
import cv2
import numpy as np
import sys

# Set input and output video paths.
input_video_path = r"input.mp4"
output_video_path = r"output.mp4"

# Improve one gray frame before detection.
def adaptive_enhance(gray):
    h, w = gray.shape[:2]
    enhanced = gray.copy()
    # Check brightness, contrast, and sharpness.
    mean_val, std_val = np.mean(gray), np.std(gray)
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()

    # Add contrast when needed.
    if mean_val < 90 or std_val < 40:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        enhanced = clahe.apply(enhanced)

    # Pick an odd filter size.
    k_size = max(3, int(w * 0.005) | 1) # Ensure odd
    # Reduce noise while keeping edges.
    enhanced = cv2.bilateralFilter(enhanced, d=k_size, sigmaColor=25, sigmaSpace=25)

    # Sharpen blurry frames.
    if sharpness < 50:
        blur = cv2.GaussianBlur(enhanced, (3,3), 0)
        enhanced = cv2.addWeighted(enhanced, 1.5, blur, -0.5, 0)
    return enhanced

# Create a mask for likely damage.
def segment_frame(gray):
    h, w = gray.shape[:2]
    mask = np.zeros_like(gray)
    # Keep only the road area.
    roi = np.array([[(0, h), (0, int(h*0.5)), (int(w*0.1), int(h*0.3)), 
                     (int(w*0.9), int(h*0.3)), (w, int(h*0.5)), (w, h)]], np.int32)
    cv2.fillPoly(mask, roi, 255)

    masked = cv2.bitwise_and(gray, mask)
    # Use road pixels to set the threshold.
    road_pixels = gray[mask == 255]
    if len(road_pixels) == 0: return mask
    
    mu, sigma = np.mean(road_pixels), np.std(road_pixels)
    dyn_thresh = np.clip(int(mu - (np.clip(sigma/25.0, 0.8, 1.8) * sigma)), 10, 180)

    # Turn dark damage areas white.
    _, binary = cv2.threshold(masked, dyn_thresh, 255, cv2.THRESH_BINARY_INV)
    binary = cv2.bitwise_and(binary, mask)
    # Clean small noise.
    k_blur = max(3, int(w * 0.003) | 1)
    return cv2.medianBlur(binary, k_blur)

# Label each detected damage area.
def classify_damage(img, mask):
    h, w = img.shape[:2]
    total_pixels = h * w
    # Set size limits from frame size.
    noise_threshold, small_crack_limit = total_pixels * 0.00003, total_pixels * 0.0003
    pothole_min, alligator_min = total_pixels * 0.0015, total_pixels * 0.001

    # Find separate damage regions.
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    overlay, detection = img.copy(), img.copy()

    for cnt in contours:
        # Measure region size.
        area = cv2.contourArea(cnt)
        # Skip tiny regions.
        if area < noise_threshold: continue

        # Measure shape width and height.
        rect = cv2.minAreaRect(cnt)
        (rw, rh) = rect[1]
        if rw == 0 or rh == 0: continue
        aspect_ratio = max(rw, rh) / min(rw, rh)
        if area < small_crack_limit and aspect_ratio < 2.5: continue

        # Measure shape solidity and roundness.
        hull = cv2.convexHull(cnt)
        solidity = area / cv2.contourArea(hull) if cv2.contourArea(hull) > 0 else 0
        h_circ = (4 * np.pi * cv2.contourArea(hull)) / (cv2.arcLength(hull, True)**2) if cv2.arcLength(hull, True) > 0 else 0

        # Measure spread of the region.
        data = cnt.reshape(-1, 2).astype(np.float64)
        spread_ratio = 0
        if len(data) > 3:
            _, _, evals = cv2.PCACompute2(data, mean=None)
            spread_ratio = evals[1][0] / evals[0][0] if evals[0][0] > 0 else 0

        # Choose color from damage type.
        if area > pothole_min and h_circ > 0.5 and solidity > 0.4:
            color, draw_cnt = (0, 0, 255), hull # Red
        elif area > alligator_min and spread_ratio > 0.1 and solidity < 0.65:
            color, draw_cnt = (255, 0, 255), cnt # Purple
        else:
            color, draw_cnt = (255, 0, 0), cnt # Blue

        # Draw the damage on the frame.
        cv2.drawContours(overlay, [draw_cnt], -1, color, -1)
        cv2.drawContours(detection, [draw_cnt], -1, color, 2)

    return cv2.addWeighted(overlay, 0.35, detection, 0.65, 0)

# Open the input video.
cap = cv2.VideoCapture(input_video_path)
if not cap.isOpened():
    print("Error opening video")
    sys.exit()

# Read video size and speed.
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Create the output video writer.
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (w, h))

print("Processing video stream...")
frame_idx = 0

# Process each video frame.
while True:
    ret, frame = cap.read()
    if not ret: break

    # Convert to gray for processing.
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # Enhance the frame.
    enhanced_gray = adaptive_enhance(gray)
    
    # Segment possible damage.
    mask = segment_frame(enhanced_gray)
    
    # Draw damage labels on the original frame.
    final_frame = classify_damage(frame, mask)
    
    # Save the processed frame.
    out.write(final_frame)
    
    frame_idx += 1
    # Show progress every 30 frames.
    if frame_idx % 30 == 0:
        print(f"Processed {frame_idx} frames...")

# Close video files.
cap.release()
out.release()
print(f"Process Complete! Final video saved to: {output_video_path}")

Processing video stream...
Processed 30 frames...
Processed 60 frames...
Processed 90 frames...
Processed 120 frames...
Processed 150 frames...
Processed 180 frames...
Processed 210 frames...
Processed 240 frames...
Processed 270 frames...
Processed 300 frames...
Processed 330 frames...
Processed 360 frames...
Processed 390 frames...
Processed 420 frames...
Processed 450 frames...
Processed 480 frames...
Processed 510 frames...
Processed 540 frames...
Processed 570 frames...
Processed 600 frames...
Processed 630 frames...
Processed 660 frames...
Processed 690 frames...
Processed 720 frames...
Processed 750 frames...
Processed 780 frames...
Processed 810 frames...
Processed 840 frames...
Processed 870 frames...
Processed 900 frames...
Processed 930 frames...
Processed 960 frames...
Process Complete! Final video saved to: C:\Users\KAVINDU\Desktop\Final_Road_Damage_Detection.mp4
